# Coursework vs. self-assessment: which classes separate the groups?

Scratch work. Question being explored:

> Do the courses people have taken differ depending on how they rate their own
> comfort (2-5) and proficiency (beg / int / adv) in programming, math, and stats?

**Two findings up front that shape everything below**, both verified before writing this:

1. **Raw "most common class per group" is uninformative.** Almost everyone in the class
   has taken the core courses (PSTAT 126: 50/51, PSTAT 120: 49/51, PSTAT 131: 46/51,
   PSTAT 122: 43/51). So the top-5 list comes back nearly identical for *every* comfort
   group. Section 2 shows this happening; Section 3 fixes it by comparing each group's
   rate against the class-wide baseline instead of ranking raw counts.

2. **Strict "this course appears only in advanced / only in beginner" returns nothing
   usable.** Section 4 runs it. The only courses that are exclusive to a level are the
   1-2 person free-text entries (LING 104, LING 110, CS 130, GEOG 186). Every real
   course was taken by at least one person at every proficiency level. Section 5 replaces
   exclusivity with a graded version -- *rate* of each course by level, ranked by how much
   it differs -- which is the answerable form of the same idea.

Keeping Section 4 in on purpose: "we checked for exclusive courses and there are none"
is a legitimate thing to report, and it stops anyone else re-deriving it later.

## Data limitations to respect throughout

- **n = 51** background respondents (49 also consented to the interest section, 55 took
   the survey overall). State which denominator every number uses.
- **`prog.comf` has exactly one person at 2.** Any "group of 2s" is one student. Cells
   below collapse 2 and 3 into a single low group; the raw version is shown first so the
   problem is visible rather than quietly handled.
- **`math.prof` and `stat.prof` have no beginners at all** (30 int / 21 adv, and
   33 adv / 18 int). "Which class is only in the beginner group" is *undefined* for
   math and stats, and rests on 3 people for programming.
- **Nothing here is causal.** Cross-sectional, self-reported, no temporal ordering
   between taking a course and rating yourself. Write it up as "is associated with" /
   "differs by", never "leads to" or "improves".
- **Selection runs both ways.** A confident programmer is more likely to elect CS 16;
   CS 16 may also build confidence. The data cannot separate these.
- Course-series labels mean *at least one* course in the series, not completion.
- Do not report a percentage of any group smaller than ~10 without the raw count beside it.

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.width', 120)
pd.set_option('display.max_columns', 40)

# Relative path lookup so this runs for everyone, from scripts/ or from the repo root.
CANDIDATES = [Path('data'), Path('../data'), Path.cwd().parent / 'data']
DATA = next((p for p in CANDIDATES if p.is_dir()), None)
if DATA is None:
    raise FileNotFoundError(
        'Could not locate the data/ folder. Run this notebook from the repo root '
        'or from scripts/.'
    )
print('Reading from:', DATA.resolve())

In [ ]:
background = pd.read_csv(DATA / 'background-clean.csv')

# Proficiency is ordinal. Store the order once so every groupby and plot below
# comes out beg -> int -> adv instead of alphabetical (adv -> beg -> int).
PROF_LEVELS = ['beg', 'int', 'adv']
PROF_COLS = ['prog.prof', 'math.prof', 'stat.prof']
COMF_COLS = ['prog.comf', 'math.comf', 'stat.comf']

for col in PROF_COLS:
    background[col] = pd.Categorical(background[col], categories=PROF_LEVELS, ordered=True)

print('background respondents:', len(background))
background[PROF_COLS + COMF_COLS].apply(lambda s: s.value_counts(dropna=False).sort_index())

### Reshaping `courses` into a person-by-course indicator table

`courses` is one comma-separated cell per respondent. Splitting on commas is safe *here*
(course labels contain no internal commas) -- but note it is **not** safe for
`interest-clean.csv`'s `areas` column, where two option labels contain commas
("Predictive modeling, generally" and "Statistical models and inference, generally").
Whoever takes the interest questions needs to match whole labels instead of splitting.

`took` ends up as 51 rows x one 0/1 column per course. Column means are the proportion
of respondents who took each course, which is what every comparison below is built on.

In [ ]:
long_courses = (
    background[['response_id', 'courses']]
    .assign(course=lambda d: d['courses'].str.split(','))
    .explode('course')
    .assign(course=lambda d: d['course'].str.strip())
    .dropna(subset=['course'])
    .loc[lambda d: d['course'] != '']
    .drop(columns='courses')
    .drop_duplicates()  # guard against a course listed twice in one cell
)

took = (
    long_courses.assign(flag=1)
    .pivot(index='response_id', columns='course', values='flag')
    .fillna(0)
    .astype(int)
)

# Traits indexed the same way, so everything can be joined by ID rather than row order.
traits = background.set_index('response_id')[PROF_COLS + COMF_COLS + [
    'upper_div_courses', 'domain_specialization', 'research'
]]
panel = traits.join(took)

print('took:', took.shape, '(respondents x courses)')
assert took.index.equals(traits.index), 'ID mismatch between courses and traits'
took.head()

## 1. Baseline: how common is each course overall?

This table is the reference point for everything after it. A course with 50/51 takers
cannot distinguish any two groups -- there is no comparison group to speak of. Read the
`usable` column as "is there enough variation here to compare takers to non-takers".

In [ ]:
N = len(took)
LOW, HIGH = 0.10, 0.90  # keep courses where both groups have roughly 5+ people

baseline = (
    pd.DataFrame({'n_took': took.sum(), 'rate': took.mean()})
    .assign(n_did_not=lambda d: N - d['n_took'])
    .assign(usable=lambda d: (d['rate'] >= LOW) & (d['rate'] <= HIGH))
    .sort_values('n_took', ascending=False)
)

# The courses with real variation. Every later comparison is restricted to these.
VARIABLE_COURSES = baseline.index[baseline['usable']].tolist()
print(f'{len(VARIABLE_COURSES)} of {len(baseline)} courses have usable variation:')
print(VARIABLE_COURSES)
baseline

## 2. The literal version: most common classes within each comfort group

This is "most common classes among people who put 5s, then 4s, then 3s". Run it to see
the problem rather than take my word for it: the top-5 lists come back essentially the
same in every group, because the top 5 courses are near-universal. The lists are telling
us about the major's requirements, not about comfort.

In [ ]:
def top_courses_by_group(group_col, k=5, courses=None):
    """Most-frequently-taken courses within each level of group_col (raw counts)."""
    sub = took if courses is None else took[courses]
    out = {}
    for level, idx in panel.groupby(group_col, observed=True).groups.items():
        counts = sub.loc[idx].sum().sort_values(ascending=False)
        label = f'{level} (n={len(idx)})'
        out[label] = [f'{c} [{counts[c]}/{len(idx)}]' for c in counts.head(k).index]
    return pd.DataFrame(out, index=[f'#{i + 1}' for i in range(k)])


for col in COMF_COLS:
    print(f'--- {col}: top 5 courses by raw count ---')
    print(top_courses_by_group(col).to_string())
    print()

## 3. The informative version: within-group rate vs. the class baseline

Same question, asked so the answer can vary. For each course and each comfort level,
compute the share of that group who took it, then subtract the class-wide rate from
Section 1. Positive means over-represented in that group.

Two changes from Section 2:

- restricted to `VARIABLE_COURSES`, since the near-universal ones have nothing to say;
- comfort collapsed to **low (2-3) / 4 / 5**, because `prog.comf == 2` is a single person
  and a one-person "group" produces rates of 0% or 100% that mean nothing.

An n of 51 split three ways means one person moves a rate by 3-10 points. Treat anything
under ~15 points as noise, and look for *ordered* patterns (low < 4 < 5) rather than a
single group standing out.

In [ ]:
COMF_BANDS = ['2-3', '4', '5']


def comfort_band(series):
    """Collapse comfort 2 and 3 together; 2 alone is n=1 for programming."""
    banded = series.map({2.0: '2-3', 3.0: '2-3', 4.0: '4', 5.0: '5'})
    return pd.Categorical(banded, categories=COMF_BANDS, ordered=True)


for col in COMF_COLS:
    panel[col + '.band'] = comfort_band(panel[col])


def rate_by_group(group_col, courses=VARIABLE_COURSES):
    """Course-by-level table of within-group take rates, plus the group sizes."""
    sizes = panel.groupby(group_col, observed=True).size()
    rates = (
        took[courses]
        .join(panel[group_col])
        .groupby(group_col, observed=True)
        .mean()
        .T
    )
    rates.columns = [f'{lvl} (n={sizes[lvl]})' for lvl in rates.columns]
    return rates, sizes


for col in COMF_COLS:
    rates, sizes = rate_by_group(col + '.band')
    table = rates.copy()
    table['overall'] = baseline.loc[VARIABLE_COURSES, 'rate']
    # Gap between the most- and least-comfortable bands: the thing worth eyeballing.
    table['gap_5_minus_low'] = rates.iloc[:, -1] - rates.iloc[:, 0]
    print(f'--- {col}: take rate within comfort band (restricted to variable courses) ---')
    print(table.sort_values('gap_5_minus_low', ascending=False).round(2).to_string())
    print()

## 4. Strict exclusivity -- the dead end, documented

"Which class shows up only in the advanced group, which only in beginner." Run it and the
result is four singleton free-text entries. Nothing here is reportable as a finding about
coursework; it is reportable as a *methodological* note.

Also worth saying plainly in the write-up: for `math.prof` and `stat.prof` there is no
beginner group to be exclusive to, and for `prog.prof` the beginner group is 3 people.

In [ ]:
def exclusive_courses(group_col):
    """Courses taken by >=1 person at this level and by nobody at any other level."""
    rows = []
    groups = panel.groupby(group_col, observed=True).groups
    for level, idx in groups.items():
        here, elsewhere = took.loc[idx], took.loc[panel.index.difference(idx)]
        exclusive = took.columns[(here.sum() > 0) & (elsewhere.sum() == 0)]
        for course in exclusive:
            rows.append({
                'group': group_col,
                'level': str(level),
                'course': course,
                'n_takers': int(here[course].sum()),
                'level_size': len(idx),
            })
    return rows


exclusivity = pd.DataFrame(
    [row for col in PROF_COLS for row in exclusive_courses(col)]
)
print('Group sizes:')
for col in PROF_COLS:
    print(' ', col, panel[col].value_counts().reindex(PROF_LEVELS).to_dict())
print()
print('Courses exclusive to a single proficiency level:')
print(exclusivity.to_string(index=False) if len(exclusivity) else '  (none)')

## 5. Graded replacement: which courses most differentiate proficiency levels?

Instead of "only in advanced", ask "**how much more often** do advanced raters report this
course than intermediate raters". Same intuition, survives a sample this size, and gives
a ranking rather than an empty set.

`int` vs `adv` is the comparison with usable cells in all three domains. Programming's
3 beginners are shown in the rate table but excluded from the difference column -- a rate
out of 3 people is not a number to rank on.

In [ ]:
def differentiation(prof_col, courses=VARIABLE_COURSES, low='int', high='adv'):
    """Rate of each course among `high` vs `low` raters, ranked by the difference."""
    sizes = panel[prof_col].value_counts()
    rates = (
        took[courses]
        .join(panel[prof_col])
        .groupby(prof_col, observed=True)
        .mean()
        .T
    )
    out = pd.DataFrame({
        f'rate_{low}': rates[low],
        f'rate_{high}': rates[high],
        f'n_{low}_takers': took.loc[panel[prof_col] == low, courses].sum(),
        f'n_{high}_takers': took.loc[panel[prof_col] == high, courses].sum(),
        'difference': rates[high] - rates[low],
    })
    print(f'{prof_col}: {low} n={sizes.get(low, 0)}, {high} n={sizes.get(high, 0)}')
    return out.sort_values('difference', ascending=False).round(2)


for col in PROF_COLS:
    print(f'--- {col}: courses ranked by (adv rate - int rate) ---')
    print(differentiation(col).to_string())
    print()

## 6. Flip it: takers vs. non-takers of each course

Sections 3 and 5 condition on the self-assessment and look at courses. This conditions on
the course and looks at the self-assessment, which is usually the more natural sentence
for a write-up: *"among the 28 respondents who had taken PSTAT 100, X% rated themselves
advanced in statistics, compared with Y% of the 23 who had not."*

Same numbers viewed differently -- pick whichever direction reads better and use it
consistently. Counts are printed alongside every rate on purpose.

In [ ]:
def takers_vs_not(course, prof_col):
    """Share rating themselves `adv` among takers vs non-takers of one course."""
    flag = took[course] == 1
    adv = panel[prof_col] == 'adv'
    return {
        'course': course,
        'outcome': prof_col,
        'n_takers': int(flag.sum()),
        'pct_adv_takers': round(100 * adv[flag].mean(), 1),
        'n_non_takers': int((~flag).sum()),
        'pct_adv_non_takers': round(100 * adv[~flag].mean(), 1),
    }


comparison = pd.DataFrame([
    takers_vs_not(course, prof_col)
    for prof_col in PROF_COLS
    for course in VARIABLE_COURSES
]).assign(gap=lambda d: d['pct_adv_takers'] - d['pct_adv_non_takers'])

# Mean comfort for the same split, since comfort is numeric and 2-5.
comfort_comparison = pd.DataFrame([
    {
        'course': course,
        'outcome': comf_col,
        'n_takers': int((took[course] == 1).sum()),
        'mean_comf_takers': round(panel.loc[took[course] == 1, comf_col].mean(), 2),
        'mean_comf_non_takers': round(panel.loc[took[course] == 0, comf_col].mean(), 2),
    }
    for comf_col in COMF_COLS
    for course in VARIABLE_COURSES
]).assign(gap=lambda d: d['mean_comf_takers'] - d['mean_comf_non_takers'])

print('--- % rating themselves advanced, takers vs non-takers ---')
print(comparison.sort_values('gap', ascending=False).to_string(index=False))
print()
print('--- mean comfort, takers vs non-takers ---')
print(comfort_comparison.sort_values('gap', ascending=False).to_string(index=False))

## 7. One figure to look at

A course-by-group heatmap of take rates. Useful for *spotting* candidates; not a
report-ready figure. If something here survives Section 5, rebuild it in R for the qmd
(`report.qmd` is R/tidyverse) as a small dot plot with the counts labelled -- a heatmap
with n=51 behind it invites readers to over-read faint colour differences.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 6), sharey=True)

for ax, col in zip(axes, PROF_COLS):
    rates, sizes = rate_by_group(col)
    im = ax.imshow(rates.values, aspect='auto', vmin=0, vmax=1, cmap='viridis')
    ax.set_xticks(range(rates.shape[1]))
    ax.set_xticklabels(rates.columns, rotation=45, ha='right')
    ax.set_yticks(range(rates.shape[0]))
    ax.set_yticklabels(rates.index)
    ax.set_title(col)
    # Print the rate in each cell; colour alone is too easy to over-interpret.
    for i in range(rates.shape[0]):
        for j in range(rates.shape[1]):
            ax.text(j, i, f'{rates.values[i, j]:.2f}', ha='center', va='center',
                    color='white', fontsize=8)

fig.colorbar(im, ax=axes, label='proportion of group who took the course', shrink=0.8)
fig.suptitle('Course take rate by self-rated proficiency (n=51, exploratory)')
plt.show()

## 8. What this is and isn't good for

**Safe to write up:**

- The coursework profile of this class is highly uniform -- the core sequence is nearly
  universal, so only a handful of electives (PSTAT 100, 160, 134, 174, CS 16) carry any
  information about differences between students. That is a finding about the sample and
  it frames everything else.
- No course is exclusive to any proficiency level (Section 4). Worth one sentence.
- Whichever elective shows the largest, *ordered* gap in Section 5 / 6 -- stated as an
  association, with counts, and with the selection caveat attached.

**Not safe:**

- Any claim that a course raised someone's proficiency or comfort.
- Any statement about beginners in math or stats. There are none.
- Ranking all 9 variable courses against 3 outcomes and reporting the winner. That is
  27 comparisons on 51 people; the largest gap will look impressive by construction.
  If we go down this road, say how many comparisons were made.

**Next steps for whoever picks this up:** pre-commit to 2-3 (course, outcome) pairs
*before* reading Section 5 closely, then report those pairs whether or not they show
anything. Decide it at the team meeting so it is on the record.

## 9. Proposed 5-way work split

This notebook covers roughly one person's slice. Suggested division (README asks for
task assignment and notes that individual grades depend on the commit history, so
everyone should commit their own script):

| # | Role | Owns | Deliverable |
|---|---|---|---|
| 1 | Coursework vs. self-assessment | this notebook; Sections 3, 5, 6 | the 2-3 pre-committed (course, outcome) pairs, as a table |
| 2 | Preparation index | `upper_div_courses`, count of electives, `research` | an ordered preparation measure + whether it tracks proficiency/comfort |
| 3 | Interests | `interest-clean.csv`: `domains`, `areas`, `language` | area/domain frequencies **with whole-label matching**, not comma splits |
| 4 | Sample description | `class-summary.csv`, `response-timing.csv`, `survey-metadata.csv` | the Data description section: how collected, who responded, 55/51/49 denominators |
| 5 | Write-up owner | `results/report.qmd` | assembles findings, renders, pushes; runs the review pass below |

Sequencing: 4 can start immediately and needs no results from anyone. 1, 2, 3 run in
parallel. 5 starts from 4's output while waiting, then integrates.

**Review pass for owner 5, before the final render:**

- every figure has a caption and a sentence saying what it shows;
- every number states its denominator (55 / 51 / 49);
- no causal verbs anywhere -- search the text for "affect", "effect", "cause", "improve",
  "lead to", "contribute";
- no percentage reported for a group under ~10 without the raw count;
- `report.html` is a render of the *current* qmd.

**Two coordination items:** the report template is R/tidyverse while this notebook and
Sid's are Python. Simplest resolution is to keep the notebooks as the exploratory record
(they are supposed to be messy scratch work) and rebuild only the final 3-4 figures in R
inside the qmd. Also: `scripts/questions.txt` currently has four causally-worded
questions; the template asks for 2-3, phrased descriptively. Rewriting that file is a
quick, visible commit for whoever wants it.